In [5]:
import pandas as pd 
import requests

In [6]:
# Jeff Sackmann's ATP data
base_url = "https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/"
years = range(2020, 2025)

for year in years:
    url = f"{base_url}atp_matches_{year}.csv"
    response = requests.get(url)
    if response.status_code == 200:
        with open(f"../data/raw/atp_matches_{year}.csv", "wb") as f:
            f.write(response.content)
        print(f"Downloaded {year} data")
    else:
        print(f"Failed to download {year} data")

Downloaded 2020 data
Downloaded 2021 data
Downloaded 2022 data
Downloaded 2023 data
Downloaded 2024 data


In [7]:
# Rankings for ELO calculation baseline
rankings_files = [
    "atp_rankings_current.csv",
    "atp_rankings_10s.csv", 
    "atp_rankings_20s.csv"
]

for file in rankings_files:
    url = f"{base_url}{file}"
    response = requests.get(url)
    if response.status_code == 200:
        with open(f"../data/raw/{file}", "wb") as f:
            f.write(response.content)
        print(f"Downloaded {file}")
    else:
        print(f"Failed to download {file}")

Downloaded atp_rankings_current.csv
Downloaded atp_rankings_10s.csv
Downloaded atp_rankings_20s.csv


In [8]:
# Load and examine data structure
matches_2024 = pd.read_csv("../data/raw/atp_matches_2024.csv")
print("Columns:", matches_2024.columns.tolist())
print("Shape:", matches_2024.shape)
print("Date range:", matches_2024['tourney_date'].min(), "to", matches_2024['tourney_date'].max())

# Check for key fields
key_fields = ['winner_name', 'loser_name', 'winner_rank', 'loser_rank', 
              'surface', 'tourney_level', 'score']

print("\n=== MISSING VALUE ANALYSIS ===")
for field in key_fields:
    missing_count = matches_2024[field].isnull().sum()
    missing_pct = (missing_count / len(matches_2024)) * 100
    print(f"{field}: {missing_count} missing ({missing_pct:.1f}%)")

# Analyze missing value patterns
print("\n=== MISSING VALUE PATTERNS ===")
print("Missing rankings by tournament level:")
missing_by_level = matches_2024.groupby('tourney_level')['winner_rank'].apply(lambda x: x.isnull().sum())
print(missing_by_level)

print("\nSample of unranked winners (these are normal):")
unranked_winners = matches_2024[matches_2024['winner_rank'].isnull()]['winner_name'].value_counts().head()
print(unranked_winners)

# Check for critical missing data
critical_missing = matches_2024[['winner_name', 'loser_name', 'surface']].isnull().sum()
print(f"\n=== CRITICAL FIELDS CHECK ===")
print("These should be near zero:")
print(critical_missing)

# Strategy documentation
print("=== MISSING VALUE STRATEGY ===")
print("""
NORMAL MISSING VALUES (Expected in tennis data):
- Rankings: Qualifiers, wildcards, young players may be unranked
- Prize money: Not always reported
- Player stats: Some tournaments don't track detailed statistics

HANDLING STRATEGY:
- Unranked players → Assign ranking of 999
- Missing surface → Drop matches (surface is critical for modeling)
- Missing names → Drop matches (impossible to model without player identity)
- Create 'is_ranked' boolean features for additional model information

This approach maintains data authenticity while enabling robust modeling.
""")

# Basic missing value summary
missing_summary = {
    'total_matches': len(matches_2024),
    'complete_matches': len(matches_2024.dropna(subset=['winner_name', 'loser_name', 'surface'])),
    'unranked_player_matches': matches_2024[['winner_rank', 'loser_rank']].isnull().any(axis=1).sum(),
    'missing_surface': matches_2024['surface'].isnull().sum()
}

print(f"\n=== DATA QUALITY SUMMARY ===")
for key, value in missing_summary.items():
    print(f"{key}: {value}")

retention_rate = (missing_summary['complete_matches'] / missing_summary['total_matches']) * 100
print(f"Data retention rate: {retention_rate:.1f}%")

Columns: ['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry', 'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age', 'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand', 'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of', 'round', 'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced', 'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points']
Shape: (3076, 49)
Date range: 20240101 to 20241218

=== MISSING VALUE ANALYSIS ===
winner_name: 0 missing (0.0%)
loser_name: 0 missing (0.0%)
winner_rank: 17 missing (0.6%)
loser_rank: 40 missing (1.3%)
surface: 0 missing (0.0%)
tourney_level: 0 missing (0.0%)
score: 0 missing (0.0%)

=== MISSING VALUE PATTERNS ===
Missing rankings by tournament level:
tou